# 08 · Verify a number against its SEC filing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/valuein/valuein/blob/main/examples/notebooks/08_verify_a_number.ipynb)

**What you will learn**

- What a `fact_id` is, and how to recompute it yourself.
- How to go from any number to the exact filing and document it came from on sec.gov.
- How to report a number you believe is wrong.

**Plan required:** none (free sample tier, no API key).

Why this matters: in research, compliance and AI workflows the question is not only "what is
the number" but "where does it come from". Every value in `fact` carries the filing that
reported it, so a reviewer can check it without trusting the data vendor.

In [1]:
%pip install -q "valuein-sdk>=7.0.0"

Note: you may need to restart the kernel to use updated packages.


## 1. Pick a number

Apple's net income for its latest fiscal year, latest vintage.

In [2]:
import pandas as pd

from valuein_sdk import FactRecord, ValueinClient, compute_fact_id

client = ValueinClient()

facts = client.read_table("fact", tickers="AAPL")
annual = facts[(facts["standard_concept"] == "NetIncome") & (facts["fiscal_period"] == "FY")]
row = annual.sort_values(["period_end", "accepted_at", "priority"]).iloc[-1]
row[["fact_id", "entity_id", "accession_id", "concept", "period_start", "period_end", "unit",
     "numeric_value", "accepted_at"]]


Upgrade access:
  Register (free)  → Benchmark: full history for every S&P 500 constituent
  Add API token    → Full US equities universe + complete history

Get your token: https://valuein.biz
  client = ValueinClient()


fact_id          9b68163632e6f1ea651ff5091c478c6e694a764b51a5ee...
entity_id                                               0000320193
accession_id                                  0000320193-25-000079
concept                                      us-gaap:NetIncomeLoss
period_start                                   2024-09-29 00:00:00
period_end                                     2025-09-27 00:00:00
unit                                                           USD
numeric_value                                       112010000000.0
accepted_at                              2025-10-31 10:01:26-04:00
Name: 2038, dtype: object

## 2. The `fact_id` is reproducible

`fact_id` is the SHA-256 hash of five identity fields:
`entity_id | accession_id | concept | period_end | unit`. It names one reported number: this
company, in this filing, for this XBRL concept, period and unit. Anyone can recompute it, so an
ID quoted in a report, a spreadsheet or an AI answer can be checked to point at a real filing.

The hash covers the identity, not the value. To check the value itself, open the filing
(section 3).

In [3]:
recomputed = compute_fact_id(row["entity_id"], row["accession_id"], row["concept"],
                             row["period_end"].date(), row["unit"])
print("fact_id from the data :", row["fact_id"])
print("recomputed            :", recomputed)
print("match                 :", recomputed == row["fact_id"])

fact_id from the data : 9b68163632e6f1ea651ff5091c478c6e694a764b51a5eef6d1f2eb053822782d
recomputed            : 9b68163632e6f1ea651ff5091c478c6e694a764b51a5eef6d1f2eb053822782d
match                 : True


`FactRecord` is a typed (Pydantic) model of one fact. `verify()` performs the same check, which
is convenient when facts travel between systems as JSON.

In [4]:
record = FactRecord(
    fact_id=row["fact_id"], entity_id=row["entity_id"], accession_id=row["accession_id"],
    concept=row["concept"], standard_concept=row["standard_concept"],
    period_end=row["period_end"].date(), accepted_at=row["accepted_at"].to_pydatetime(),
    numeric_value=row["numeric_value"], unit=row["unit"],
)
print("FactRecord.verify():", record.verify())

FactRecord.verify(): True


## 3. Open the filing

`accession_id` is the SEC's filing number. The `filing` table gives its form, dates and URL;
`client.filing_links()` adds direct links to the document and to the SEC's Inline XBRL viewer,
where every tagged number is highlighted. Search the viewer for the concept name
(`us-gaap:NetIncomeLoss` here) or for the value.

In [5]:
filing = client.run_query(f"""
    SELECT accession_id, form_type, filing_date, accepted_at, report_date, filing_url
    FROM filing
    WHERE accession_id = '{row["accession_id"]}'
""")
filing.T

,0
accession_id,0000320193-25-000079
form_type,10-K
filing_date,2025-10-31 00:00:00
accepted_at,2025-10-31 10:01:26-04:00
report_date,2025-09-27 00:00:00
filing_url,https://www.sec.gov/Archives/edgar/data/320193...


In [6]:
links = client.filing_links("AAPL", form_types=["10-K"], limit=5)
match = links[links["accession_id"] == row["accession_id"]].iloc[0]
best = match["inline_viewer_url"] or match["document_url"] or match["viewer_url"] or match["sec_url"]
print("Open this to check the number:", best)
print("Filing index (every document):", match["sec_url"])

Open this to check the number: https://www.sec.gov/ix?doc=/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm
Filing index (every document): https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/0000320193-25-000079-index.htm


Prefer `inline_viewer_url`, then `document_url`, `viewer_url` and `sec_url`. The Inline XBRL
viewer only exists for filings made in that format (phased in for 10-K and 10-Q filers from 2019), so
for older filings `inline_viewer_url` is empty and the fallbacks apply.

## 4. Report a number you believe is wrong

`client.report_issue()` sends a report to the Valuein team. It works with no API key. Include
the `fact_id` and ticker so the filing can be located. This cell is not executed in the
published notebook, because running it files a real report:

```python
receipt = client.report_issue(
    "Net income for FY2025 does not match the 10-K income statement",
    category="data_quality",
    fact_id=row["fact_id"],
    ticker="AAPL",
    expected_value="<the value you see in the filing>",
)
print(receipt.feedback_id, receipt.status)
```

## 5. The same check from an AI assistant

The Valuein MCP server exposes `verify_fact_lineage`, which takes a `fact_id` and returns the
filing behind it. An assistant connected to the MCP server can cite a `fact_id` for every number
it states, and a human can check any of them with this notebook. See notebook 09.

In [7]:
client.close()

## Next steps

- **[09 · Use Valuein from an AI assistant](09_ai_assistant.ipynb)**.
- Script version: [`python/filing_provenance.py`](../python/filing_provenance.py).